# Local SME Underwriting Agents Notebook

In [ ]:
from __future__ import annotations

import json
import os
import sys

from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv


PROJECT_ROOT = Path().resolve()
load_dotenv(PROJECT_ROOT / ".env", override=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
TESTCASE_ID = "case_1"

# Chọn trên màn hình upload. Để rỗng thì dùng DEFAULT_LOAN_PROGRAM (PLO).
# B1CP, MISA, PLPP, PLO
LOAN_PROGRAM = "PLO"

# Agent chọn trên màn hình upload: BUSINESS_ACTIVITY_AGENT,
# FINANCIAL_ANALYSIS_AGENT, CREDIT_RELATIONSHIP_AGENT, CREDIT_PROPOSAL_AGENT
AGENT = "FINANCIAL_ANALYSIS_AGENT"

# CSV/MD/PDF/TXT/XLS/XLSX/XML
INPUT_PATHS = [
    str(PROJECT_ROOT / "samples" / TESTCASE_ID),
    # "/absolute/path/to/BCTC.pdf",
]

OUTPUT_DIR = PROJECT_ROOT / "logs"
MAX_CHARS_PER_DOCUMENT = 120_000

In [ ]:
from src.config import Config, build_llm

config = Config(
    document_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.5
    ),
    analysis_llm=build_llm(
        "MODEL_ANALYZER",
        temperature=0.1,
    ),
    financial_statement_extraction_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.0,
    ),
    proposal_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    cic_s10a_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    cic_r21_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    sitevisit_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    ledger_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)

In [ ]:
from src.types import (
    AgentName,
    WorkflowMode,
    DocumentAgentName,
    ClassifiedDocument,
    UnderwritingGraphState,
    to_dict_list,
    extract_text_from_agent_output,
    truncate_text,
)

In [ ]:
from src.agents.documents.document_classification import (
    document_type_scores,
    rule_classify_document,
)
from src.agents.documents.document_discovery import (
    compute_file_hash,
    discover_documents,
    resolve_input_path,
)
from src.utils.common import (
    SUPPORTED_EXTENSIONS,
    normalize_text,
)
from src.types import VALID_DOCUMENT_AGENTS
from src.matrix.document_matrix import (
    agent_relevance_for_type,
    load_matrix,
    primary_agent_for_type,
)

_matrix = load_matrix()
print(
    f"Document matrix v{_matrix.version}: {len(_matrix.types)} document types \n"
    f"Loan programs: {', '.join(_matrix.loan_programs)}"
)


In [ ]:
from src.agents.specialist import (
    SpecialistAgent,
    BusinessActivityAnalysis,
    FinancialAnalysis,
    CreditRelationshipAnalysis,
    CreditProposalAnalysis,
)
from src.agents.supervisor import Supervisor

In [ ]:
from IPython.display import Markdown, display
from src.utils.common import show_graph
from src.agents.supervisor import EXTRACTION_PASSES

supervisor = Supervisor(config)

display(Markdown("### Workflow Graph"))
display(show_graph(supervisor.workflow_graph))

result = supervisor.process(
    INPUT_PATHS, agent=AGENT, loan_program=LOAN_PROGRAM
)

display(Markdown(result["response"]))

print("\n--- Agent Name ---")
print(result["agent_name"])

print("\n--- Steps ---")
for step in result["steps"]:
    print("-", step)

print("\n--- Document Classifications ---")
classification_keys = [
    "filename",
    "document_type",
    "declared_group",
    "document_group",
    "source_description",
    "agent_relevance",
    "loan_program",
    "agent",
    "confidence",
    "reasoning",
    "extraction_status",
    "extraction_error",
    "is_financial_statement",
    "financial_statement_extraction_error",
    "is_proposal",
    "proposal_extraction_error",
    "is_cic_s10a",
    "cic_s10a_extraction_error",
    "is_cic_r21",
    "cic_r21_extraction_error",
    "is_sitevisit",
    "sitevisit_extraction_error",
    "is_ledger",
    "ledger_extraction_error",
    "classifier_error_type",
    "classifier_error",
]
for item in result["document_classifications"]:
    classification = {key: item.get(key) for key in classification_keys}
    print(json.dumps(classification, ensure_ascii=False, indent=2))

# Derived from EXTRACTION_PASSES rather than retyped: the prefixes, flags and
# labels used to be listed here by hand, which is the drift the table exists to
# prevent — and it now also has to know which passes are load-bearing.
_failures = [
    (item["filename"], pass_, item.get(pass_.error_attr) or "Không rõ lý do.")
    for item in result["document_classifications"]
    for pass_ in EXTRACTION_PASSES
    if item.get(pass_.flag_attr) and not item.get(pass_.result_attr)
]
_blocking = [(n, p_, w) for n, p_, w in _failures if p_.required]
_degrading = [(n, p_, w) for n, p_, w in _failures if not p_.required]

if result["agent_name"] == "EXTRACTION_FAILED":
    print(
        f"\nDỪNG: {len(_blocking)} tài liệu bắt buộc trích xuất thất bại. "
        "Không có báo cáo — hệ thống KHÔNG dùng OCR thô thay thế:"
    )
    for _name, _pass, _why in _blocking:
        print(f"  - {_name} ({_pass.label}): {_why}")
    print("  Xử lý nguyên nhân ở trên rồi chạy lại.")
elif _blocking:
    # Should not happen: a required failure ends the run. Printed rather than
    # ignored so a gap between the two rules is visible instead of silent.
    print(
        f"\nWARNING: {len(_blocking)} trích xuất bắt buộc thất bại nhưng "
        "luồng vẫn chạy — kiểm tra ExtractionPass.required:"
    )
    for _name, _pass, _why in _blocking:
        print(f"  - {_name} ({_pass.label}): {_why}")

if _degrading:
    print(
        f"\nWARNING: {len(_degrading)} lần trích xuất thất bại ở pass không "
        "bắt buộc — agent dùng OCR thô cho các tài liệu này:"
    )
    for _name, _pass, _why in _degrading:
        print(f"  - {_name} ({_pass.label}): {_why}")

_unmatched = [
    item["filename"]
    for item in result["document_classifications"]
    if not item.get("document_type")
]
if _unmatched:
    print(
        f"\nWARNING: {len(_unmatched)} document(s) matched no type in the "
        f"matrix and were shared with every agent: {', '.join(_unmatched)}"
    )


In [ ]:
run_id = "_".join([TESTCASE_ID, datetime.now().strftime("%Y%m%d_%H%M%S")])
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

from src.utils.report.visualization.report_html import embed_diagrams

post_monitor_content = [
    {
        'filename': 'result.json',
        'content': result
    },
    {
        'filename': 'document_classifications.json',
        'content': result["document_classifications"]
    },
    {
        'filename': 'document_selections.json',
        'content': result["document_selections"]
    },
    {
        'filename': 'financial_metrics.json',
        'content': result["financial_metrics"]
    },
    {
        'filename': 'credit_need.json',
        'content': result["credit_need"]
    },
    {
        'filename': 'financial_statement_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'financial_statement_extraction',
        'condition': 'is_financial_statement' 
    },
    {
        'filename': 'proposal_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'proposal_extraction',
        'condition': 'is_proposal' 
    },
    {
        'filename': 'cic_s10a_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'cic_s10a_extraction',
        'condition': 'is_cic_s10a' 
    },
    {
        'filename': 'cic_r21_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'cic_r21_extraction',
        'condition': 'is_cic_r21' 
    },
    {
        'filename': 'sitevisit_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'sitevisit_extraction',
        'condition': 'is_sitevisit' 
    },
    {
        'filename': 'ledger_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'ledger_extraction',
        'condition': 'is_ledger' 
    }
]

(run_dir / "final_response.md").write_text(
    embed_diagrams(result["response"]), 
    encoding="utf-8"
)

# result.json used to repeat almost every file beside it — the OCR text, each
# pass's extraction JSON, the metrics, the credit need, the selections. On a
# 22-file dossier that put the payload past the export limit on the far side.
# The exclusion sets are derived from post_monitor_content below rather than
# retyped, so adding a sibling file cannot leave result.json duplicating it.
_own_file_keys = {
    c['filename'].removesuffix('.json')
    for c in post_monitor_content
    if not c.get('content_lv2') and c['filename'] != 'result.json'
}
_own_file_doc_keys = {
    c['content_lv2'] for c in post_monitor_content if c.get('content_lv2')
}

for c in post_monitor_content:
    if c['filename'] == 'result.json':
        # Written last, after every sibling has landed, so nothing is dropped
        # here that failed to arrive there.
        continue
    if c.get('content_lv2', None):
        (run_dir / c['filename']).write_text(
            json.dumps(
                {
                    doc["filename"]: doc.get(c['content_lv2'])
                    for doc in c['content']
                    if doc.get(c['condition'])
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
    else:
        (run_dir / c['filename']).write_text(
            json.dumps(c['content'], ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

_slim_result = {
    key: value
    for key, value in result.items()
    if key not in _own_file_keys and key != 'document_classifications'
}
_slim_result['document_classifications'] = [
    {k: v for k, v in doc.items() if k not in _own_file_doc_keys}
    for doc in result['document_classifications']
]
_slim_result['artifacts'] = (
    'Extraction results, metrics, credit need and document selections are in '
    'the sibling JSON files in this directory. OCR text is truncated here to '
    'Config.result_content_char_limit; set OCR_CACHE_DIR for the full text.'
)
(run_dir / 'result.json').write_text(
    json.dumps(_slim_result, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(
    f"result.json {len(json.dumps(_slim_result, ensure_ascii=False, indent=2)):,} chars "
    f"(was {len(json.dumps(result, ensure_ascii=False, indent=2)):,} before de-duplication)"
)

# Export the final response (Markdown) to PDF via `markdown` + WeasyPrint.
import platform

if platform.system() == "Darwin":
    for brew_lib in ("/opt/homebrew/lib", "/usr/local/lib"):
        if os.path.isdir(brew_lib):
            os.environ["DYLD_LIBRARY_PATH"] = (
                brew_lib + ":" + os.environ.get("DYLD_LIBRARY_PATH", "")
            )

from weasyprint import HTML
from src.utils.report.visualization.report_html import build_report_html

html_doc = build_report_html(result["response"])
HTML(string=html_doc).write_pdf(str(run_dir / "final_response.pdf"))

print(f"Saved PDF: {run_dir / 'final_response.pdf'}")
print(f"Saved artifacts to: {run_dir}")